# Weak-Reader Probe Evaluation - H11 bar completion

**Author**: Knowledge Graph Foundry autonomous build

Answers the 28-probe set with Claude Haiku 4.5 as the reader (orchestrator stays untouched for extraction elsewhere), propositions off vs on, on the densified R01 graph. Both arms share the R03 serialization (head+tail, citations) and abstention gate, so the toggle isolates H11's proposition context. The weak-reader claim: propositions let a small model answer what it otherwise misses.

In [1]:
# imports
import datetime  # stamps
import json  # persistence
import re  # scoring
from pathlib import Path  # paths

import yaml  # probe set

from knowledge_graph_foundry import Foundry, LLMSettings, load_settings

2026-07-06 13:30:30.666 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [2]:
# configuration
PROBES_PATH = Path('../tests/probes/cpap-probe-set.yml')
REPORTS = Path('../reports')
WEAK_MODEL = 'bedrock/eu.anthropic.claude-haiku-4-5-20251001-v1:0'  # reader under test
probes = yaml.safe_load(PROBES_PATH.read_text())
print(len(probes), 'probes; weak reader:', WEAK_MODEL)

28 probes; weak reader: bedrock/eu.anthropic.claude-haiku-4-5-20251001-v1:0


In [3]:
# scoring (same deterministic rules as probe_eval.ipynb)
REFUSAL = re.compile(
    r'no information|not (?:available|stated|specified|mentioned|provided)|'
    r'lacks|does not (?:contain|include|specify|state|provide|mention)|'
    r'cannot answer|unable to|unanswerable|no (?:data|details|answer)|'
    r'does not contain enough information', re.I)

def _norm(s):
    return re.sub(r'\s+', ' ', s.casefold())

def value_tokens(gold_answer):
    return [t for t in re.findall(r'[\w.\-/]*\d[\w.\-/]*', gold_answer)]

def answer_correct(probe, answer):
    ans = _norm(answer)
    if probe['category'] == 'unanswerable':
        return bool(REFUSAL.search(answer))
    tokens = value_tokens(probe['gold_answer'])
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ans)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    return _norm(probe['gold_answer']) in ans

def run_weak(settings, label):
    rows = []
    with Foundry(settings) as f:
        for p in probes:
            answer = f.query(p['question'])['answer']
            rows.append({'id': p['id'], 'category': p['category'],
                         'correct': answer_correct(p, answer), 'answer': answer})
            print(p['id'], p['category'], rows[-1]['correct'])
    answerable = [r for r in rows if r['category'] != 'unanswerable']
    unans = [r for r in rows if r['category'] == 'unanswerable']
    return {'label': label, 'rows': rows,
            'answer_accuracy': sum(r['correct'] for r in answerable) / len(answerable),
            'correct_refusal': sum(r['correct'] for r in unans) / len(unans)}

In [4]:
# weak reader, propositions OFF
s0 = load_settings()
s0.llm = LLMSettings(engine='frontier', model=WEAK_MODEL, region='eu-central-1')
s0.graphrag.propositions_enabled = False
w0 = run_weak(s0, 'W0_weak_noprops')
print(w0['label'], w0['answer_accuracy'], w0['correct_refusal'])

2026-07-06 13:30:31.079 | WARNING  | knowledge_graph_foundry.settings:load_settings:148 - config file config.yml not found, using defaults


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 13:30:31.378 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 1/1
2026-07-06 13:30:31.379 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 1/1 entities via bedrock (0 cache hits)
2026-07-06 13:30:31.412 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:188 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


P01 single_fact True


...

P27 unanswerable True


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 13:31:34.384 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 1/1
2026-07-06 13:31:34.385 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 1/1 entities via bedrock (0 cache hits)
2026-07-06 13:31:34.420 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:188 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


P28 unanswerable True
W0_weak_noprops 0.7083333333333334 1.0


In [5]:
# weak reader, propositions ON
s1 = load_settings()
s1.llm = LLMSettings(engine='frontier', model=WEAK_MODEL, region='eu-central-1')
s1.graphrag.propositions_enabled = True
w1 = run_weak(s1, 'W1_weak_props')
print(w1['label'], w1['answer_accuracy'], w1['correct_refusal'])

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 13:31:36.218 | WARNING  | knowledge_graph_foundry.settings:load_settings:148 - config file config.yml not found, using defaults
2026-07-06 13:31:36.219 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:31:36.237 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:188 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


...

P26 unanswerable True
2026-07-06 13:32:25.786 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:32:25.798 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:188 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


P27 unanswerable True
2026-07-06 13:32:27.970 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:32:27.984 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:188 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}


P28 unanswerable True
W1_weak_props 0.75 1.0


In [6]:
# summary + persist
for w in (w0, w1):
    print(f"{w['label']:<18} acc={w['answer_accuracy']:.3f} refusal_ok={w['correct_refusal']:.2f}")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
out = REPORTS / f'probe-eval-weak-{stamp}.json'
out.write_text(json.dumps({'W0': w0, 'W1': w1}, indent=2, default=str))
print('saved', out)

W0_weak_noprops    acc=0.708 refusal_ok=1.00
W1_weak_props      acc=0.750 refusal_ok=1.00
saved ../reports/probe-eval-weak-20260706-113229.json
